<a href="https://colab.research.google.com/github/laramalkawi81-ops/DS230-Instacart-Project/blob/main/Copy_of_Untitled24.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Feature Engineering & Model Preparation

This notebook focuses on constructing user-level, product-level, and user–product interaction features, as well as preparing time-aware datasets for downstream modeling tasks.

In [2]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [3]:
DATA_PATH = '/content/drive/MyDrive/instacart_data/instacart_data'

orders = pd.read_csv(f"{DATA_PATH}/orders_clean.csv")
order_products_prior = pd.read_csv(f"{DATA_PATH}/order_products_prior_clean.csv")
products = pd.read_csv(f"{DATA_PATH}/products.csv")



In [4]:
orders.shape, order_products_prior.shape

((3421083, 8), (32434489, 5))

Merge orders with order_products

20% sample

In [5]:
sample_users = orders['user_id'].drop_duplicates().sample(frac=0.2, random_state=42)


sample_orders = orders[orders['user_id'].isin(sample_users)]
sample_prior = order_products_prior[order_products_prior['order_id'].isin(sample_orders['order_id'])]

print(sample_orders.shape)
print(sample_prior.shape)

(687880, 8)
(6550242, 5)


In [6]:
data = sample_prior.merge(
    sample_orders,
    on='order_id',
    how='left'
)

print(data.shape)
data.head()


(6550242, 12)


,order_id,product_id,add_to_cart_order,reordered,product_freq,user_id,eval_set,order_number,order_dow,order_hour_of_day,days_since_prior_order,days_since_prior_order_scaled
0,6,40462,1,0,0.000009,22352,prior,4,1,12,30.0,2.100730
1,6,15873,2,0,0.000002,22352,prior,4,1,12,30.0,2.100730
2,6,41897,3,0,0.000001,22352,prior,4,1,12,30.0,2.100730
3,28,35108,1,0,0.000591,98256,prior,29,3,13,6.0,-0.477496
4,28,40593,2,1,0.000215,98256,prior,29,3,13,6.0,-0.477496


User-level Features

In [7]:
user_features = sample_orders.groupby('user_id').agg(
    total_orders=('order_number', 'max'),
    mean_days_between_orders=('days_since_prior_order', 'mean')
).reset_index()

user_features.head()

,user_id,total_orders,mean_days_between_orders
0,5,5,9.200000
1,7,21,9.952381
2,13,13,7.076923
3,23,5,14.800000
4,32,6,18.500000


User-level features summarize overall shopping behavior, including ordering frequency and average reorder timing.

Product-level Features

In [8]:
product_features = sample_prior.groupby('product_id').agg(
    product_reorder_rate=('reordered', 'mean'),
    product_purchase_count=('order_id', 'count')
).reset_index()

product_features.head()


,product_id,product_reorder_rate,product_purchase_count
0,1,0.596667,300
1,2,0.000000,13
2,3,0.837500,80
3,4,0.424242,66
4,5,0.250000,4


Product-level features capture global popularity and reorder tendencies across all users.

User × Product Interaction Features

In [9]:
user_product_features = data.groupby(['user_id', 'product_id']).agg(
    user_product_count=('order_id', 'count'),
    user_product_reorder_rate=('reordered', 'mean')
).reset_index()

user_product_features.head()

,user_id,product_id,user_product_count,user_product_reorder_rate
0,5,3376,1,0.00
1,5,5999,1,0.00
2,5,6808,1,0.00
3,5,8518,2,0.50
4,5,11777,4,0.75


merge all features

In [10]:
data = data.merge(user_features, on='user_id', how='left')
data = data.merge(product_features, on='product_id', how='left')
data = data.merge(user_product_features, on=['user_id', 'product_id'], how='left')

data.head()
data.shape

(6550242, 18)

Temporal Features

In [11]:
data['order_dow'] = data['order_dow']
data['order_hour'] = data['order_hour_of_day']


Temporal features such as order day of week and hour of day capture recurring behavioral patterns in shopping activity.

Classification Target

In [12]:
target = 'reordered'
data[target].value_counts(normalize=True)



,proportion
reordered,
1,0.590826
0,0.409174


Time-aware Split

In [13]:
train_data = data[data['order_number'] < data['order_number'].quantile(0.8)]
test_data  = data[data['order_number'] >= data['order_number'].quantile(0.8)]

train_data.shape, test_data.shape

((5238960, 19), (1311282, 19))

A time-aware split was used to prevent data leakage by ensuring that training data precedes validation data chronologically.

In this notebook, we engineered features at multiple levels: user-level, product-level, and user–product interactions. We combined behavioral and temporal information and prepared time-aware training and testing datasets for supervised learning.